---

# 📚 Aula 2 - Pipeline de Processamento de Texto com NLTK

> **Objetivo:** reproduzir com a biblioteca NLTK as etapas do pipeline trabalhado na Aula 1:  
> **tokenização → stopwords → stemming**.

⚠️ Observação: o NLTK depende do download de alguns recursos (corpora). No Google Colab, isso é comum.


## 🧰 Instalação e downloads necessários


In [ ]:
# Instalação (Colab). Em ambientes locais, pode não ser necessário.
!pip -q install nltk


In [ ]:
import nltk

nltk.download("punkt") # Necessário para a tokenização de texto em palavras e sentenças
nltk.download("stopwords") # Contém listas de palavras comuns (stopwords) para remoção
nltk.download("punkt_tab") # Recurso para tokenização de texto em português, incluindo separação de palavras e sentenças.
nltk.download('rslp') # Necessário para o algoritmo RSLP (Removedor de Sufixos da Língua Portuguesa) de stemming.

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package rslp to /root/nltk_data...
[nltk_data]   Unzipping stemmers/rslp.zip.


True

## 🧠 Pipeline completo
1.	Higienização do texto com expressões regulares  
2.	Normalização com operações sobre Unicode  
3.	Tokenização com NLTK  
4.	Remoção de stopwords com NLTK, ajustada conforme a tarefa  
5.	Redução morfológica (Stemming – NLTK)

In [ ]:
import re
import unicodedata
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import RSLPStemmer

def higienizar(texto: str) -> str:
    # remove marcações HTML
    texto = re.sub(r"<[^>]+>", " ", texto)
    # remove URLs
    texto = re.sub(r"http\S+|www\.\S+", "", texto)
    # remove pontuação e símbolos (mantém letras/números/_ e espaços)
    texto = re.sub(r"[^\w\s]", " ", texto, flags=re.UNICODE)
    # normaliza espaços
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

def normalizar(texto: str, remover_acentos: bool = True) -> str:
    texto = texto.lower()
    if remover_acentos:
        texto = unicodedata.normalize("NFD", texto)
        texto = "".join(c for c in texto if not unicodedata.combining(c))
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

def pipeline_nltk(
    texto_bruto: str,
    tarefa: str = "sentimentos",
    remover_acentos: bool = True
):
    # 1) Higienização
    texto_limpo = higienizar(texto_bruto)

    # 2) Normalização
    texto_norm = normalizar(texto_limpo, remover_acentos=remover_acentos)

    # 3) Tokenização (NLTK)
    tokens = word_tokenize(texto_norm, language="portuguese")

    # 4) Stopwords (NLTK)
    stop_pt = set(stopwords.words("portuguese"))
    negacao = "nao" if remover_acentos else "não"

    if tarefa == "sentimentos":
        tokens_filtrados = [t for t in tokens if (t not in stop_pt) or (t == negacao)]
    else:
        tokens_filtrados = [t for t in tokens if t not in stop_pt]

    # 5) Redução morfológica (Stemming – NLTK)
    stemmer = RSLPStemmer()
    tokens_reduzidos = [stemmer.stem(t) for t in tokens_filtrados]

    return {
        "texto_bruto": texto_bruto,
        "texto_limpo": texto_limpo,
        "texto_normalizado": texto_norm,
        "tokens": tokens,
        "tokens_filtrados": tokens_filtrados,
        "tokens_reduzidos": tokens_reduzidos
    }


In [ ]:
# ======================
# Demonstração (exemplo)
# ======================
texto_base = "Não gostei... o produto veio com defeito 😡 http://exemplo.com"

saida = pipeline_nltk(texto_base, tarefa="sentimentos", remover_acentos=True)

print("Texto bruto:        ", saida["texto_bruto"])
print("Após higienização:  ", saida["texto_limpo"])
print("Após normalização:  ", saida["texto_normalizado"])
print("Tokens (NLTK):      ", saida["tokens"])
print("Sem stopwords:      ", saida["tokens_filtrados"])
print("Redução morfológica:", saida["tokens_reduzidos"])

Texto bruto:         Não gostei... o produto veio com defeito 😡 http://exemplo.com
Após higienização:   Não gostei o produto veio com defeito
Após normalização:   nao gostei o produto veio com defeito
Tokens (NLTK):       ['nao', 'gostei', 'o', 'produto', 'veio', 'com', 'defeito']
Sem stopwords:       ['nao', 'gostei', 'produto', 'veio', 'defeito']
Redução morfológica: ['nao', 'gost', 'produt', 'vei', 'defeit']
